### Exercise #4 - correction

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import seaborn as sns # For plotting
import matplotlib.pyplot as plt # For showing plots
from statsmodels.graphics.gofplots import qqplot

#### Exercise 4.1

Using an appropriate visualization, check the effect of Mean Annual Temperature (Temp_ann) on the presence of Salmo trutta fario (Brown Trout).

Import and clean data

In [ ]:
# Import the dataset - adapt the path to your local environment
df = pd.read_csv('../Examples-local/EFIplus_medit.zip',compression='zip', sep=";")

In [ ]:
# clean up the dataset to remove unnecessary columns (eg. REG) 
df.drop(df.iloc[:,5:15], axis=1, inplace=True)

# let's rename some columns so that they make sense
df.rename(columns={'Sum of Run1_number_all':'Total_fish_individuals'}, inplace=True) # inplace="True" means that df will be updated

# for sake of consistency, let's also make all column labels of type string
df.columns = list(map(str, df.columns))

In [ ]:
df2 = df.dropna() # drop all rows with missing values (NA)

Some possible visualization settings

Strip plot

In [ ]:
sns.stripplot(data=df, y='Salmo trutta fario', x='temp_ann', orient='h',
    jitter=0.1, # the ammount of jitter (i.e. random point dispersion along the x-axis).
    linewidth=0, # no line around the points
    color='#9e2a2b', # HEX color picked from here: https://coolors.co/palettes/trending - don't forget to start by "#"
    alpha=.3, # transparency of the points (between 0 and 1)
    s=2, # point size
    )
plt.ylabel('Salmo trutta fario', fontdict={'size': 12, 'style': 'italic'})
plt.xlabel('Mean Annual Temperature')
plt.show()

In [ ]:
sns.set_theme(rc={'figure.figsize':(4,5)}) # Resize figure
sns.set_style(style='white') # Set the background of the plot to white

sns.boxplot(
    data=df,
    x='Salmo trutta fario',
    y='temp_ann',
    notch=True,
    hue='Salmo trutta fario',        # fixes palette warning
    palette=['lightgray', 'skyblue'],
    width=0.4,
    linewidth=1,
    legend=False
)

plt.xticks(ticks=[0, 1], labels=['Absence', 'Presence'])  # safe way

plt.xlabel('Brown trout occurrence', fontsize=12)
plt.ylabel('Mean Annual Temperature', fontsize=12)

plt.show()


In [ ]:
sns.set_theme(rc={'figure.figsize':(4,5)}) # Resize figure
sns.set_style(style='white') # Set the background of the plot to white

# Create boxplots with a notch
sns.boxplot(data=df, x='Salmo trutta fario', y='temp_ann', # x and y are the variables to be plotted
        notch=True, # add a notch to the boxplot to show the confidence interval around the median
        hue='Salmo trutta fario', # add hue to avoid palette warning
        palette=['lightgray', 'skyblue'], # color palette for the two categories (absence and presence)
        width=0.4, # width of the boxes
        linewidth=1, # width of the lines around the boxes
        legend=False,               # prevents duplicate legend
        )
plt.xticks(ticks=[0, 1], labels=['Absence', 'Presence'])  # set x-tick labels
plt.xlabel('Brown trout occurrence', fontdict={'size': 12}) # x-axis label
plt.ylabel('Mean Annual Temperature', fontdict={'size': 12}) # y-axis label

plt.show()

Histograms

In [ ]:
# To restore matplotlib settings:
import matplotlib as mpl
mpl.rc_file_defaults()

In [ ]:
# Plot histograms of mean annual temperature for sites with and without brown trout
sns.histplot(data=df[df['Salmo trutta fario']==0], x='temp_ann', color='gray', alpha=0.5, edgecolor=None, label="Absence of Brown trout")
sns.histplot(data=df[df['Salmo trutta fario']==1], x='temp_ann', alpha=0.4, edgecolor=None, label="Presence of Brown trout")
plt.legend(frameon=False) # add legend without frame
plt.xlabel('Mean Annual Temperature') # x-axis label
plt.show()


In [ ]:
# Plot density plots of mean annual temperature for sites with and without brown trout
sns.kdeplot(data=df[df['Salmo trutta fario']==0], x='temp_ann', color='gray', fill=True, label="Absence of Brown trout")
sns.kdeplot(data=df[df['Salmo trutta fario']==1], x='temp_ann', fill=True, label="Presence of Brown trout")
plt.legend(frameon=False, loc='upper left')
plt.xlabel('Mean Annual Temperature')
plt.show()

Violin plots

In [ ]:
sns.violinplot(data=df, x='Salmo trutta fario', y='temp_ann')
plt.ylim(0, 20)
plt.show()

Simple scatter plot

In [ ]:
sns.scatterplot(data=df, x='Salmo trutta fario', y='temp_ann')
plt.ylim(0, 20)
plt.show()

#### Exercise 4.2
Check the same effect but now separately for Minho and in the Tagus catchments and comparing the “effect sizes”.


In [ ]:
df_selec = df[(df['Catchment_name'] == 'Minho') | (df['Catchment_name'] == 'Tejo')] # select only the rows corresponding to the Minho and Tejo catchments

# Create violin plots of mean annual temperature for the Minho and Tejo catchments, separated by brown trout occurrence
sns.violinplot(
    data=df_selec,
    x='Catchment_name',
    y='temp_ann',
    hue='Salmo trutta fario',
)
plt.xlabel('')
plt.ylabel('Mean Annual Temperature')
plt.show()


#### Exercise 4.3
Test, using both visualization and hypothesis testing methods, if the actual_river_slope is drawn from a normal distribution.

In [ ]:
# Plot histogram of actual river slope
sns.histplot(df2['Actual_river_slope'])
plt.show()

Seems to be very right skewed

In [ ]:
from statsmodels.graphics.gofplots import qqplot # Import the qqplot function from the statsmodels library

# Create a Q-Q plot of the actual river slope variable to assess its normality
qqplot(pd.Series(df2['Actual_river_slope']), line='s')
plt.show()

The QQ plot shows that the actual river slope variable is not normally distributed, as the points deviate from the straight line.

Compute the Sapiro-wilk test for normality of the actual river slope variable

In [ ]:
# import function
from scipy.stats import shapiro

df2 = df.dropna() # drops rows when at least one element is a missing value
df2.info()

# normality test
stat, p = shapiro(pd.Series(df2['Actual_river_slope']))
print('Statistics=%.3f, p=%.3f' % (stat, p)) # print outputs
# interpret. H0: 'the sample was drawn from a Gaussian distribution'.
alpha = 0.05
if p > alpha:
 print('Sample is not significantly different from Gaussian (fail to reject H0. Rejecting H0 has an error probability >0.05)')
else:
 print('Sample is significantly different from Gaussian (reject H0 with an error probability <0.05)')

#### Exercise 4.4

Take 100 samples of 2000 observations with replacement, compute the mean for each sample and plot the resulting histogram of means. Test if these 100 mean values are drawn from a normal distribution.


In [ ]:
mean = [] # create an empty list to store the means of the samples
for i in range(0,100): # repeat the process of sampling and calculating the mean 100 times
    slope = df['Actual_river_slope'] # select the 'Actual_river_slope' column from the dataframe to be used for sampling
    sampler = np.random.randint(0, len(slope), 2000) # generate 2000 random integer numbers with replacement to be used as random indices
    sample = slope.take(sampler) # take 2000 random observations from slope
    mean.append(sample.mean()) # calculate the mean of the sample and append it to the list of means

sns.histplot(mean, kde=True) # plot a histogram of the means of the samples with a kernel density estimate (KDE) to show the distribution of the means
plt.show()

Q-Q plot:

In [ ]:
# Create a Q-Q plot of the means of the samples to assess their normality
qqplot(pd.Series(mean), line='s')
plt.show()

Shapiro-wilk test

In [ ]:
# import function
from scipy.stats import shapiro
from scipy.stats import kstest

# Shapiro-Wilk normality test 
stat, p = shapiro(mean) # perform the Shapiro-Wilk test on the means of the samples 
print('Shapiro-Wilk normality test:')
print('Statistics=%.3f, p=%.3f' % (stat, p)) # print outputs
# interpret. H0: 'the sample was drawn from a Gaussian distribution'.
alpha = 0.05 # set significance level
if p > alpha:
 print('H0 is not rejected (Rejecting H0 has an error probability >0.05)')
else:
 print('reject H0 (Rejecting H0 has an error probability <0.05)')